# 6 自定义量化 A8W8 matmul 算子开发并接入 Qwen3-8B

本章是整个启航营的**核心实践环节**，将学习到的所有知识沉淀转化为一次完整的自定义算子开发实战。

实践的环境准备、提交方式可参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

### 本章定位

本章将开发一个自定义的量化 matmul 算子 `QmmCustom`，通过 A8W8 量化（Activation INT8 + Weight INT8）减少计算量和内存占用，从而提升推理性能。这是本次启航营中综合性最强的一章——涵盖了从算子概念理解、Tiling 设计、Kernel 实现到算子编译、接入网络与功能性能测试的全链路。

### 通过实践达成的学习目标

- **理解量化推理原理**：掌握 A8W8 量化方案（INT8 激活 + INT8 权重）的数学原理与 Scale 反量化机制，理解 perChannelScale 与 perTokenScale 的作用。
- **设计 Tiling 数据结构与函数**：根据算子原型定义自行设计 TilingData 结构体，实现多核分块的 Tiling 函数，理解 Cube 计算单元的分块策略。
- **实现 Cube+Vector 双路径 Kernel**：完成 Cube-only（INT32 输出）和 Cube+Vector 协作（反量化 BF16 输出）两条 Kernel 路径的实现，理解昇腾 AI Core 的 Cube 与 Vector 协作编程模型。
- **完成算子编译与 Torch 接入**：编译自定义算子并通过 Torch 接口接入，打通“算子开发 → 框架调用”的链路。
- **测试算子功能与性能，尝试优化迭代**：验证自定义算子的精度正确性，测试单算子与网络性能，并根据测试结果对 Tiling 或 Kernel 进行优化迭代。

### 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 自定义算子实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译自定义算子
    - **任务四：成功编译自定义算子**
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
- 提交要求
---

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_tutorial_dir():
    cwd = Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.append(base)
        candidates.append(base / 'tutorials/llm_inference/qwen3_8b')
        candidates.append(base / 'cann-learning-hub/tutorials/llm_inference/qwen3_8b')
    for candidate in candidates:
        if ((candidate / 'src/op_custom/qmm_custom').is_dir()
                and (candidate / 'src/inference_scripts/recipe_qwen3_8b').is_dir()):
            return candidate
    raise FileNotFoundError('Cannot locate the qwen3_8b tutorial directory.')

TUTORIAL_DIR = locate_tutorial_dir()
REPO_ROOT = TUTORIAL_DIR
os.chdir(TUTORIAL_DIR)
os.environ['TUTORIAL_DIR'] = str(TUTORIAL_DIR)
TUTORIAL_SRC_DIR = TUTORIAL_DIR / 'src'
if str(TUTORIAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_SRC_DIR))

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Source the CANN environment first.')
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env = subprocess.check_output(
    ['bash', '-lc', f'source {cann_script} && env'], text=True, cwd=TUTORIAL_DIR
)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

SRC_DIR = TUTORIAL_SRC_DIR / 'op_custom'
QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'
RECIPE_ROOT = TUTORIAL_SRC_DIR / 'inference_scripts/recipe_qwen3_8b'
os.environ['RECIPE_ROOT'] = str(RECIPE_ROOT)

remote_w8a8 = Path('/mnt/workspace/qwen3_8b_models/Qwen3-8B-W8A8')
default_w8a8 = TUTORIAL_DIR / 'src/data/models/Qwen3-8B-W8A8'
if remote_w8a8.is_dir():
    os.environ.setdefault('QWEN3_W8A8_WEIGHT', str(remote_w8a8))
else:
    os.environ.setdefault('QWEN3_W8A8_WEIGHT', str(default_w8a8))

print('CANN 包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
print('教程目录 =', TUTORIAL_DIR)
print('算子目录 =', QMM_CUSTOM_DIR)
print('recipes 目录 =', RECIPE_ROOT)
print('W8A8 权重 =', os.environ['QWEN3_W8A8_WEIGHT'])
print('[环境准备] 完成', flush=True)


[环境准备] 开始


CANN 包路径 = /home/developer/Ascend/cann-9.0.0
教程目录 = /mnt/workspace/qwen3_8b_tutorial_final/v1
算子目录 = /mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/qmm_custom
recipes 目录 = /mnt/workspace/qwen3_8b_tutorial_final/v1/src/inference_scripts/recipe_qwen3_8b
W8A8 权重 = /mnt/workspace/qwen3_8b_models/Qwen3-8B-W8A8
[环境准备] 完成


安装所需的依赖：

In [2]:
%pip install -r {SRC_DIR}/requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


Note: you may need to restart the kernel to use updated packages.


---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

### 2.2 算子规格

在上一章中，通过分析量化模型的profiling，归纳出了自定义算子的输入输出规格：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

## 3 自定义算子实现

与前面学习过的内容一致，本次实践采用 `<<<>>>` Kernel 直调方式开发上述原型的量化 matmul 算子。直调方式通过 `kernel<<<numBlocks, dynUBufSize, stream>>>(args)` 语法在 Host 侧直接启动 Kernel，适合快速验证算子功能与性能。

本节包含如下三个任务：

### 任务一：自行设计 TilingData 结构体并实现 Tiling 函数

根据算子原型与 Kernel 的实现需求，自行设计 `QmmCustomTilingData` 结构体，并实现 `CalcQmmTiling` 函数。Tiling 函数需根据输入的 M、N、K 维度，结合平台硬件资源（L1/L0A/L0B/UB 内存限制、可用核数），计算多核分块参数。可借助 `platform_ascendc::PlatformAscendCManager` 获取硬件信息。

> **Tiling 设计提示**：
> 
> 设计 TilingData 时需要考虑以下要素：
> 1. **`TCubeTiling`**：CANN Matmul 库提供的标准 Tiling 结构体，包含 M/N/K 的分块大小（`baseM`、`baseN`、`baseK`）、使用的核数（`usedCoreNum`）等字段。可通过 `GetMatmulTiling` API 自动计算。
> 2. **`isPertoken`**：标记是否走反量化路径，用于 Kernel 内部分支选择。
> 3. **`workspaceSize`**：Path 2 中 Cube 写入 INT32 中间结果所需的 GM workspace 大小。
> 4. **多核分块策略**：将 M 维度或 N 维度分配到多个核上并行计算，每个核处理一个分块。分块大小需平衡 L1/L0A/L0B 缓存利用率与多核负载均衡。

### 任务二：实现 A8W8 输入、INT32 输出的 Kernel

实现 `QmmCubeBasicKernel` 类，完成纯 Cube 路径的 INT8 矩阵乘法计算。

计算公式： ```result_int32 = x1_int8 @ x2_int8```

### 任务三：实现 A8W8 输入、经 perChannelScale 和 perTokenScale 反量化输出 BF16 的 Kernel

实现 `QmmPertokenKernel` 类，在 Cube 计算的基础上增加 Vector 反量化路径。

计算公式： ```result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale```

其中 `perChannelScale` 沿 N 维广播，`perTokenScale` 沿 M 维广播，两者均为 FLOAT32 类型。

---

### 执行路径说明

在 Host 侧的 `qmm_custom` 函数中，根据是否传入 `pertoken_scale` 设置 `isPertoken` 标志，Kernel 启动时据此选择执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1（Cube-only）**：仅使用 AI Core 的 Cube 计算单元，完成 INT8×INT8 矩阵乘法后直接将 INT32 结果写入输出 GM，无 Vector 参与。对应 `QmmCubeBasicKernel` 的实现。

**Path 2（Cube+Vector）**：Cube 单元先完成 INT8 matmul 得到 INT32 中间结果并写入 GM workspace；随后 Vector 单元从 GM 读取 INT32 结果，依次完成 `Cast`（INT32→FLOAT32）、乘以 `perChannelScale` 和 `perTokenScale`、`Cast`（FLOAT32→BF16）等反量化操作，最终将 BF16 结果写入输出 GM。对应 `QmmPertokenKernel` 的实现。

Kernel 入口函数 `qmm_custom_kernel` 使用 `__mix__(1, 2)` 属性声明，表示为每个 Block 分配 1 个 Cube 核和 2 个 Vector 核，使 Cube 与 Vector 可在同一 Kernel 中协同工作。Kernel 内部通过 `tilingData.isPertoken` 判断走哪条路径，两条路径复用相同的 Tiling 结构和 Kernel 启动入口。

Tiling、Kernel、Torch 接口的实现均位于 `qmm_custom.asc` 文件中。文件已给出整体框架与函数签名，请自行完成下方代码中的 TODO 部分，以实现完整的自定义算子：

In [3]:
%%writefile {QMM_CUSTOM_DIR}/qmm_custom.asc
#include <algorithm>
#include <cstdlib>
#include <cstdint>
#include <iostream>
#include <vector>

#include "acl/acl_base.h"
#include "acl/acl_prof.h"
#include "kernel_operator.h"
#define ASCENDC_CUBE_ONLY
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
#include "torch_npu/csrc/core/npu/NPUStream.h"
#include <torch/extension.h>

namespace {
constexpr uint32_t kMaxM = 4096U;
constexpr uint32_t kMinK = 4096U;
constexpr uint32_t kMaxK = 12288U;
constexpr uint32_t kMinN = 4096U;
constexpr uint32_t kMaxN = 24576U;
constexpr uint32_t kNzN0 = 32U;
constexpr uint32_t kSmallVectorTileCols = 256U;
constexpr uint32_t kMediumVectorTileCols = 1024U;
constexpr uint32_t kLargeVectorTileCols = 8192U;
constexpr uint32_t kVectorTileRows = 1U;
constexpr uint32_t kBf16M50N6144CoreLimit = 16U;
constexpr uint32_t kWorkspaceAlignment = 512U;
constexpr uint16_t kCubeToVectorFlag0 = 7U;
constexpr uint16_t kVectorToCubeFlag0 = 8U;
constexpr uint16_t kCubeToVectorFlag1 = 9U;
constexpr uint16_t kVectorToCubeFlag1 = 10U;
constexpr uint16_t kCubeToVectorFlag2 = 11U;
constexpr uint16_t kVectorToCubeFlag2 = 12U;
constexpr uint16_t kCubeToVectorFlag3 = 13U;
constexpr uint16_t kVectorToCubeFlag3 = 14U;
constexpr uint32_t kVectorSubcoreCount = 2U;

uint32_t CeilDiv(uint32_t value, uint32_t divisor)
{
    return (value + divisor - 1U) / divisor;
}

uint32_t AlignUp(uint32_t value, uint32_t alignment)
{
    return CeilDiv(value, alignment) * alignment;
}

uint64_t AlignUp64(uint64_t value, uint64_t alignment)
{
    return ((value + alignment - 1U) / alignment) * alignment;
}

bool IsSupportedShape(uint64_t m, uint64_t k, uint64_t n)
{
    return m >= 1U && m <= kMaxM && k >= kMinK && k <= kMaxK && n >= kMinN && n <= kMaxN &&
           (k % kNzN0 == 0U) && (n % 16U == 0U);
}

void ConfigureMatmulTiling(matmul_tiling::MultiCoreMatmulTiling &tiling, uint32_t m, uint32_t n, uint32_t k)
{
    tiling.SetAType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                    matmul_tiling::DataType::DT_INT8, false);
    tiling.SetBType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::NZ,
                    matmul_tiling::DataType::DT_INT8, false);
    tiling.SetCType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                    matmul_tiling::DataType::DT_INT32);
    tiling.SetBias(false);
    tiling.SetOrgShape(static_cast<int32_t>(m), static_cast<int32_t>(n), static_cast<int32_t>(k));
    tiling.SetShape(static_cast<int32_t>(m), static_cast<int32_t>(n), static_cast<int32_t>(k));
}
} // namespace

#pragma pack(push, 8)
struct alignas(8) QmmCustomTilingData {
    uint32_t m;
    uint32_t n;
    uint32_t k;
    uint32_t isPertoken;
    uint32_t usedCoreNum;
    uint32_t mCoreBlocks;
    uint32_t nCoreBlocks;
    uint32_t vectorTileCols;
    uint32_t vectorTileRows;
    uint32_t cubeTilingBytes;
    uint32_t workspaceBaseIsFullAllocation;
    uint32_t reserved;
    uint64_t systemWorkspaceBytes;
    uint64_t userWorkspaceBytes;
    uint64_t totalWorkspaceBytes;
    uint64_t intermediateOffsetBytes;
    uint64_t intermediateBytes;
    TCubeTiling cubeTilingData;
};
#pragma pack(pop)

static_assert(sizeof(QmmCustomTilingData) % alignof(uint64_t) == 0,
              "QmmCustom tiling data must stay 8-byte aligned");

namespace {
bool IsLegalTiling(const TCubeTiling &cubeTiling, uint32_t m, uint32_t n, uint32_t maxCoreNum)
{
    if (cubeTiling.usedCoreNum <= 0 || cubeTiling.singleCoreM <= 0 || cubeTiling.singleCoreN <= 0 ||
        static_cast<uint32_t>(cubeTiling.usedCoreNum) > maxCoreNum ||
        static_cast<uint32_t>(cubeTiling.singleCoreN) % kNzN0 != 0U) {
        return false;
    }
    const uint32_t mCoreBlocks = CeilDiv(m, static_cast<uint32_t>(cubeTiling.singleCoreM));
    const uint32_t nCoreBlocks = CeilDiv(n, static_cast<uint32_t>(cubeTiling.singleCoreN));
    return static_cast<uint64_t>(mCoreBlocks) * nCoreBlocks ==
           static_cast<uint32_t>(cubeTiling.usedCoreNum);
}

QmmCustomTilingData CalcQmmTiling(uint32_t isPertoken, uint32_t m, uint32_t n, uint32_t k)
{
    auto *platform = platform_ascendc::PlatformAscendCManager::GetInstance();
    TORCH_CHECK(platform != nullptr, "qmm_custom: failed to obtain Ascend platform information");
    const uint32_t maxCoreNum = static_cast<uint32_t>(platform->GetCoreNumAic());
    TORCH_CHECK(maxCoreNum > 0U, "qmm_custom: device reports no AIC cores");

    TCubeTiling cubeTiling = {};
    matmul_tiling::MultiCoreMatmulTiling automaticTiling(*platform);
    automaticTiling.SetDim(static_cast<int32_t>(maxCoreNum));
    ConfigureMatmulTiling(automaticTiling, m, n, k);

    // These are the v32 shape-specific base tiles for the large-M, large-N K=4096 cases.
    if (m >= 1024U && k == 4096U && n >= 6144U) {
        TORCH_CHECK(automaticTiling.SetFixSplit(256, 128, 64) == 0,
                    "qmm_custom: failed to set the v32 fixed base tile");
    }

    const bool automaticOk =
        automaticTiling.GetTiling(cubeTiling) == 0 && IsLegalTiling(cubeTiling, m, n, maxCoreNum);
    if (!automaticOk) {
        // Safe v32 fallback: split only N, and align every FRACTAL_NZ start to N0.
        uint32_t candidateLimit = maxCoreNum;
        if (isPertoken != 0U && m == 50U && n == 6144U) {
            candidateLimit = std::min(candidateLimit, kBf16M50N6144CoreLimit);
        }
        uint32_t nCoreCount = 1U;
        uint32_t singleCoreN = AlignUp(n, kNzN0);
        for (uint32_t candidate = candidateLimit; candidate > 0U; --candidate) {
            const uint32_t proposed = AlignUp(CeilDiv(n, candidate), kNzN0);
            if (CeilDiv(n, proposed) == candidate) {
                nCoreCount = candidate;
                singleCoreN = proposed;
                break;
            }
        }

        matmul_tiling::MultiCoreMatmulTiling fallbackTiling(*platform);
        fallbackTiling.SetDim(static_cast<int32_t>(nCoreCount));
        TORCH_CHECK(fallbackTiling.SetSingleShape(static_cast<int32_t>(AlignUp(m, 16U)),
                                                  static_cast<int32_t>(singleCoreN),
                                                  static_cast<int32_t>(k)) == 0,
                    "qmm_custom: failed to set the N-only fallback shape");
        ConfigureMatmulTiling(fallbackTiling, m, n, k);
        TORCH_CHECK(fallbackTiling.GetTiling(cubeTiling) == 0 &&
                        IsLegalTiling(cubeTiling, m, n, maxCoreNum),
                    "qmm_custom: vendor tiler did not produce a legal FRACTAL_NZ tiling");
    }

    QmmCustomTilingData result = {};
    result.m = m;
    result.n = n;
    result.k = k;
    result.isPertoken = isPertoken;
    result.usedCoreNum = static_cast<uint32_t>(cubeTiling.usedCoreNum);
    result.mCoreBlocks = CeilDiv(m, static_cast<uint32_t>(cubeTiling.singleCoreM));
    result.nCoreBlocks = CeilDiv(n, static_cast<uint32_t>(cubeTiling.singleCoreN));
    TORCH_CHECK(static_cast<uint64_t>(result.mCoreBlocks) * result.nCoreBlocks == result.usedCoreNum,
                "qmm_custom: usedCoreNum does not match the M/N core grid");
    if (isPertoken != 0U) {
        const uint64_t vectorBlockNum = static_cast<uint64_t>(result.usedCoreNum) * 2U;
        const uint32_t maxAivCoreNum = static_cast<uint32_t>(platform->GetCoreNumAiv());
        TORCH_CHECK(vectorBlockNum <= UINT32_MAX && vectorBlockNum <= maxAivCoreNum,
                    "qmm_custom: standalone BF16 vector phase requires ", vectorBlockNum,
                    " pure Vector blocks, but the device exposes only ", maxAivCoreNum, " AIV cores");
    }

    const uint32_t preferredVectorTileCols =
        m == 1U ? kSmallVectorTileCols : (m >= 1024U ? kLargeVectorTileCols : kMediumVectorTileCols);
    result.vectorTileCols =
        std::min(static_cast<uint32_t>(cubeTiling.singleCoreN), preferredVectorTileCols);
    result.vectorTileRows = kVectorTileRows;
    result.cubeTilingBytes = static_cast<uint32_t>(sizeof(TCubeTiling));
    result.workspaceBaseIsFullAllocation = 1U;
    result.systemWorkspaceBytes = static_cast<uint64_t>(platform->GetLibApiWorkSpaceSize());
    result.intermediateBytes =
        isPertoken != 0U ? static_cast<uint64_t>(m) * n * sizeof(int32_t) : 0U;
    result.intermediateOffsetBytes = result.systemWorkspaceBytes;
    result.userWorkspaceBytes = result.intermediateBytes;
    TORCH_CHECK(result.systemWorkspaceBytes <= UINT64_MAX - result.userWorkspaceBytes,
                "qmm_custom: workspace size overflow");
    result.totalWorkspaceBytes = result.systemWorkspaceBytes + result.userWorkspaceBytes;
    result.cubeTilingData = cubeTiling;
    return result;
}

__aicore__ inline uint32_t DeviceMin(uint32_t lhs, uint32_t rhs)
{
    return lhs < rhs ? lhs : rhs;
}

using QmmAType = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, int8_t>;
using QmmBType = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::NZ, int8_t>;
using QmmCType = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, int32_t>;
using QmmBiasType = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, int32_t>;
using QmmMatmulMdl = AscendC::Matmul<QmmAType, QmmBType, QmmCType, QmmBiasType, CFG_MDL>;
using QmmMatmulNorm = AscendC::Matmul<QmmAType, QmmBType, QmmCType, QmmBiasType, CFG_NORM>;

template <int HAS_PERTOKEN_SCALE, typename MatmulImpl>
class QmmCubeKernel {
public:
    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR y, GM_ADDR workspace,
                                const QmmCustomTilingData &tiling, uint32_t logicalCoreId)
    {
        tiling_ = &tiling;
        cubeTiling_ = tiling.cubeTilingData;
        logicalCoreId_ = logicalCoreId;
        x1Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x1),
                              static_cast<uint64_t>(tiling.m) * tiling.k);
        x2Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x2),
                              static_cast<uint64_t>(tiling.k) * tiling.n);
        GM_ADDR cubeOutput = HAS_PERTOKEN_SCALE != 0 ? workspace + tiling.intermediateOffsetBytes : y;
        outGm_.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(cubeOutput),
                               static_cast<uint64_t>(tiling.m) * tiling.n);
        REGIST_MATMUL_OBJ(&pipe_, GetSysWorkSpacePtr(), matmul_, &cubeTiling_);
    }

    __aicore__ inline void Process()
    {
        if constexpr (HAS_PERTOKEN_SCALE != 0) {
            if (UseFineGrainedPipeline()) {
                RunPipelinedTile(logicalCoreId_);
                matmul_.End();
                AscendC::CrossCoreWaitFlag(kVectorToCubeFlag0);
                AscendC::CrossCoreWaitFlag(kVectorToCubeFlag1);
                AscendC::CrossCoreWaitFlag(kVectorToCubeFlag2);
                AscendC::CrossCoreWaitFlag(kVectorToCubeFlag3);
                return;
            }
        }
        RunTile(logicalCoreId_);
        matmul_.End();
        if constexpr (HAS_PERTOKEN_SCALE != 0) {
            AscendC::CrossCoreSetFlag<0x2, PIPE_FIX>(kCubeToVectorFlag0);
            AscendC::CrossCoreWaitFlag(kVectorToCubeFlag0);
        }
    }

private:
    __aicore__ inline bool UseFineGrainedPipeline() const
    {
        return tiling_->m >= 1024U && tiling_->n >= 6144U;
    }

    __aicore__ inline void RunPipelinedTile(uint32_t tileId)
    {
        const uint32_t mBlock = tileId % tiling_->mCoreBlocks;
        const uint32_t nBlock = tileId / tiling_->mCoreBlocks;
        const uint32_t mStart = mBlock * static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM);
        const uint32_t nStart = nBlock * static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreN);
        const uint32_t tileM = DeviceMin(static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM),
                                         tiling_->m - mStart);
        const uint32_t tileN = DeviceMin(static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreN),
                                         tiling_->n - nStart);
        const uint32_t splitM1 = DeviceMin(tileM, ((tileM + 63U) / 64U) * 16U);
        const uint32_t splitM2 = DeviceMin(tileM, ((tileM + 31U) / 32U) * 16U);
        const uint32_t splitM3 = DeviceMin(tileM, ((3U * tileM + 63U) / 64U) * 16U);
        RunTileSlice(mStart, nStart, 0U, splitM1, tileN);
        AscendC::CrossCoreSetFlag<0x2, PIPE_FIX>(kCubeToVectorFlag0);
        RunTileSlice(mStart, nStart, splitM1, splitM2 - splitM1, tileN);
        AscendC::CrossCoreSetFlag<0x2, PIPE_FIX>(kCubeToVectorFlag1);
        RunTileSlice(mStart, nStart, splitM2, splitM3 - splitM2, tileN);
        AscendC::CrossCoreSetFlag<0x2, PIPE_FIX>(kCubeToVectorFlag2);
        RunTileSlice(mStart, nStart, splitM3, tileM - splitM3, tileN);
        AscendC::CrossCoreSetFlag<0x2, PIPE_FIX>(kCubeToVectorFlag3);
    }

    __aicore__ inline void RunTileSlice(uint32_t mStart, uint32_t nStart, uint32_t localRowStart,
                                       uint32_t sliceM, uint32_t tileN)
    {
        if (sliceM == 0U) {
            return;
        }
        const uint64_t x1Offset = static_cast<uint64_t>(mStart + localRowStart) * tiling_->k;
        const uint64_t x2Offset = static_cast<uint64_t>(nStart) * tiling_->k;
        const uint64_t outOffset = static_cast<uint64_t>(mStart + localRowStart) * tiling_->n + nStart;
        matmul_.SetSingleShape(static_cast<int>(sliceM), static_cast<int>(tileN),
                               static_cast<int>(tiling_->k));
        matmul_.SetTensorA(x1Gm_[x1Offset]);
        matmul_.SetTensorB(x2Gm_[x2Offset]);
        matmul_.IterateAll(outGm_[outOffset]);
    }

    __aicore__ inline void RunTile(uint32_t tileId)
    {
        const uint32_t mBlock = tileId % tiling_->mCoreBlocks;
        const uint32_t nBlock = tileId / tiling_->mCoreBlocks;
        const uint32_t mStart = mBlock * static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM);
        const uint32_t nStart = nBlock * static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreN);
        const uint32_t tileM = DeviceMin(static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM),
                                         tiling_->m - mStart);
        const uint32_t tileN = DeviceMin(static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreN),
                                         tiling_->n - nStart);
        const uint64_t x1Offset = static_cast<uint64_t>(mStart) * tiling_->k;
        // A non-transposed int8 FRACTAL_NZ N tile occupies K contiguous elements per N column.
        const uint64_t x2Offset = static_cast<uint64_t>(nStart) * tiling_->k;
        const uint64_t outOffset = static_cast<uint64_t>(mStart) * tiling_->n + nStart;
        matmul_.SetTensorA(x1Gm_[x1Offset]);
        matmul_.SetTensorB(x2Gm_[x2Offset]);
        if (tileM < static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM) ||
            tileN < static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreN)) {
            matmul_.SetTail(static_cast<int>(tileM), static_cast<int>(tileN));
        }
        matmul_.IterateAll(outGm_[outOffset]);
    }

    AscendC::TPipe pipe_;
    MatmulImpl matmul_;
    AscendC::GlobalTensor<int8_t> x1Gm_;
    AscendC::GlobalTensor<int8_t> x2Gm_;
    AscendC::GlobalTensor<int32_t> outGm_;
    const QmmCustomTilingData *tiling_ = nullptr;
    TCubeTiling cubeTiling_;
    uint32_t logicalCoreId_ = 0U;
};

class QmmVectorKernel {
public:
    __aicore__ inline void Init(GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y, GM_ADDR workspace,
                                const QmmCustomTilingData &tiling, uint32_t logicalCoreId,
                                uint32_t subBlockId)
    {
        tiling_ = &tiling;
        logicalCoreId_ = logicalCoreId;
        subBlockId_ = subBlockId;
        intermediateGm_.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(workspace + tiling.intermediateOffsetBytes),
            static_cast<uint64_t>(tiling.m) * tiling.n);
        scaleGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(scale), tiling.n);
        pertokenScaleGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(pertokenScale), tiling.m);
        yGm_.SetGlobalBuffer(reinterpret_cast<__gm__ bfloat16_t *>(y),
                             static_cast<uint64_t>(tiling.m) * tiling.n);
        pipe_.InitBuffer(int32InQueue_, 1, tiling.vectorTileCols * sizeof(int32_t));
        pipe_.InitBuffer(scaleInQueue_, 1, tiling.vectorTileCols * sizeof(float));
        pipe_.InitBuffer(bf16OutQueue_, 1, tiling.vectorTileCols * sizeof(bfloat16_t));
        pipe_.InitBuffer(calcBuffer_, 2U * tiling.vectorTileCols * sizeof(float));
        int32AsFloat_ = calcBuffer_.GetWithOffset<float>(tiling.vectorTileCols, 0U);
        columnScaled_ = calcBuffer_.GetWithOffset<float>(tiling.vectorTileCols,
                                                          tiling.vectorTileCols * sizeof(float));
    }

    __aicore__ inline void Process()
    {
        if (UseFineGrainedPipeline()) {
            const uint32_t mBlock = logicalCoreId_ % tiling_->mCoreBlocks;
            const uint32_t mStart = mBlock * static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM);
            const uint32_t tileM = DeviceMin(static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM),
                                             tiling_->m - mStart);
            const uint32_t splitM1 = DeviceMin(tileM, ((tileM + 63U) / 64U) * 16U);
            const uint32_t splitM2 = DeviceMin(tileM, ((tileM + 31U) / 32U) * 16U);
            const uint32_t splitM3 = DeviceMin(tileM, ((3U * tileM + 63U) / 64U) * 16U);
            AscendC::CrossCoreWaitFlag(kCubeToVectorFlag0);
            RunTileRows(logicalCoreId_, 0U, splitM1);
            AscendC::CrossCoreSetFlag<0x2, PIPE_MTE3>(kVectorToCubeFlag0);
            AscendC::CrossCoreWaitFlag(kCubeToVectorFlag1);
            RunTileRows(logicalCoreId_, splitM1, splitM2 - splitM1);
            AscendC::CrossCoreSetFlag<0x2, PIPE_MTE3>(kVectorToCubeFlag1);
            AscendC::CrossCoreWaitFlag(kCubeToVectorFlag2);
            RunTileRows(logicalCoreId_, splitM2, splitM3 - splitM2);
            AscendC::CrossCoreSetFlag<0x2, PIPE_MTE3>(kVectorToCubeFlag2);
            AscendC::CrossCoreWaitFlag(kCubeToVectorFlag3);
            RunTileRows(logicalCoreId_, splitM3, tileM - splitM3);
            AscendC::CrossCoreSetFlag<0x2, PIPE_MTE3>(kVectorToCubeFlag3);
            return;
        }
        AscendC::CrossCoreWaitFlag(kCubeToVectorFlag0);
        RunTile(logicalCoreId_);
        AscendC::CrossCoreSetFlag<0x2, PIPE_MTE3>(kVectorToCubeFlag0);
    }

    __aicore__ inline void ProcessStandaloneBf16VectorPhase()
    {
        RunTile(logicalCoreId_);
    }

private:
    __aicore__ inline bool UseFineGrainedPipeline() const
    {
        return tiling_->m >= 1024U && tiling_->n >= 6144U;
    }

    template <typename T>
    __aicore__ inline void CopyIn(const AscendC::LocalTensor<T> &dst,
                                  const AscendC::GlobalTensor<T> &src, uint32_t count)
    {
        const AscendC::DataCopyExtParams copyParams = {
            1U, static_cast<uint32_t>(count * sizeof(T)), 0U, 0U, 0U};
        const AscendC::DataCopyPadExtParams<T> padParams = {false, 0, 0, 0};
        AscendC::DataCopyPad<T>(dst, src, copyParams, padParams);
    }

    template <typename T>
    __aicore__ inline void CopyOut(const AscendC::GlobalTensor<T> &dst,
                                   const AscendC::LocalTensor<T> &src, uint32_t count)
    {
        const AscendC::DataCopyExtParams copyParams = {
            1U, static_cast<uint32_t>(count * sizeof(T)), 0U, 0U, 0U};
        AscendC::DataCopyPad<T>(dst, src, copyParams);
    }

    __aicore__ inline void RunTile(uint32_t tileId)
    {
        const uint32_t mBlock = tileId % tiling_->mCoreBlocks;
        const uint32_t mStart = mBlock * static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM);
        const uint32_t tileM = DeviceMin(static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM),
                                         tiling_->m - mStart);
        RunTileRows(tileId, 0U, tileM);
    }

    __aicore__ inline void RunTileRows(uint32_t tileId, uint32_t localRowStart, uint32_t rowCount)
    {
        const event_t eventIdSToV =
            static_cast<event_t>(GetTPipePtr()->FetchEventID(AscendC::HardEvent::S_V));
        const uint32_t mBlock = tileId % tiling_->mCoreBlocks;
        const uint32_t nBlock = tileId / tiling_->mCoreBlocks;
        const uint32_t mStart = mBlock * static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM);
        const uint32_t nStart = nBlock * static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreN);
        const uint32_t tileM = DeviceMin(static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreM),
                                         tiling_->m - mStart);
        const uint32_t tileN = DeviceMin(static_cast<uint32_t>(tiling_->cubeTilingData.singleCoreN),
                                         tiling_->n - nStart);
        const uint32_t localRowEnd = DeviceMin(tileM, localRowStart + rowCount);
        for (uint32_t localRow = localRowStart; localRow < localRowEnd; ++localRow) {
            if (localRow % kVectorSubcoreCount != subBlockId_) {
                continue;
            }
            const uint32_t globalRow = mStart + localRow;
            const float rowScale = pertokenScaleGm_.GetValue(globalRow);
            AscendC::SetFlag<AscendC::HardEvent::S_V>(eventIdSToV);
            AscendC::WaitFlag<AscendC::HardEvent::S_V>(eventIdSToV);
            for (uint32_t localCol = 0U; localCol < tileN; localCol += tiling_->vectorTileCols) {
                const uint32_t count = DeviceMin(tiling_->vectorTileCols, tileN - localCol);
                ProcessRowSegment(globalRow, nStart + localCol, count, rowScale);
            }
        }
    }

    __aicore__ inline void ProcessRowSegment(uint32_t row, uint32_t column, uint32_t count,
                                             float rowScale)
    {
        const uint64_t matrixOffset = static_cast<uint64_t>(row) * tiling_->n + column;
        auto int32Local = int32InQueue_.AllocTensor<int32_t>();
        auto scaleLocal = scaleInQueue_.AllocTensor<float>();
        CopyIn(int32Local, intermediateGm_[matrixOffset], count);
        CopyIn(scaleLocal, scaleGm_[column], count);
        int32InQueue_.EnQue(int32Local);
        scaleInQueue_.EnQue(scaleLocal);
        int32Local = int32InQueue_.DeQue<int32_t>();
        scaleLocal = scaleInQueue_.DeQue<float>();
        AscendC::Cast(int32AsFloat_, int32Local, AscendC::RoundMode::CAST_NONE, count);
        AscendC::PipeBarrier<PIPE_V>();
        AscendC::Mul(columnScaled_, int32AsFloat_, scaleLocal, count);
        AscendC::PipeBarrier<PIPE_V>();
        AscendC::Muls(columnScaled_, columnScaled_, rowScale, count);
        AscendC::PipeBarrier<PIPE_V>();
        auto bf16Local = bf16OutQueue_.AllocTensor<bfloat16_t>();
        AscendC::Cast(bf16Local, columnScaled_, AscendC::RoundMode::CAST_RINT, count);
        AscendC::PipeBarrier<PIPE_V>();
        bf16OutQueue_.EnQue(bf16Local);
        int32InQueue_.FreeTensor(int32Local);
        scaleInQueue_.FreeTensor(scaleLocal);
        bf16Local = bf16OutQueue_.DeQue<bfloat16_t>();
        CopyOut(yGm_[matrixOffset], bf16Local, count);
        bf16OutQueue_.FreeTensor(bf16Local);
    }

    AscendC::TPipe pipe_;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> int32InQueue_;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> scaleInQueue_;
    AscendC::TQue<AscendC::TPosition::VECOUT, 1> bf16OutQueue_;
    AscendC::TBuf<AscendC::TPosition::VECCALC> calcBuffer_;
    AscendC::GlobalTensor<int32_t> intermediateGm_;
    AscendC::GlobalTensor<float> scaleGm_;
    AscendC::GlobalTensor<float> pertokenScaleGm_;
    AscendC::GlobalTensor<bfloat16_t> yGm_;
    AscendC::LocalTensor<float> int32AsFloat_;
    AscendC::LocalTensor<float> columnScaled_;
    const QmmCustomTilingData *tiling_ = nullptr;
    uint32_t logicalCoreId_ = 0U;
    uint32_t subBlockId_ = 0U;
};
} // namespace

__global__ __mix__(1, 2) __aicore__
void qmm_custom_kernel(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale,
                       GM_ADDR y, __kfc_workspace__ GM_ADDR workspace,
                       QmmCustomTilingData tilingData)
{
    // This direct-launch build does not define HAVE_WORKSPACE. The argument is therefore the full
    // allocation base: [Matmul system workspace][BF16 int32 intermediate].
    if ASCEND_IS_AIC {
        AscendC::SetSysWorkspace(workspace);
        const uint32_t logicalCoreId = AscendC::GetBlockIdx();
        if (tilingData.isPertoken == 0U) {
            if (tilingData.m >= 1024U && tilingData.n == 4096U) {
                QmmCubeKernel<0, QmmMatmulNorm> cubeKernel;
                cubeKernel.Init(x1, x2, y, workspace, tilingData, logicalCoreId);
                cubeKernel.Process();
            } else {
                QmmCubeKernel<0, QmmMatmulMdl> cubeKernel;
                cubeKernel.Init(x1, x2, y, workspace, tilingData, logicalCoreId);
                cubeKernel.Process();
            }
        } else {
            if (tilingData.m >= 1024U && tilingData.n == 4096U) {
                QmmCubeKernel<1, QmmMatmulNorm> cubeKernel;
                cubeKernel.Init(x1, x2, y, workspace, tilingData, logicalCoreId);
                cubeKernel.Process();
            } else {
                QmmCubeKernel<1, QmmMatmulMdl> cubeKernel;
                cubeKernel.Init(x1, x2, y, workspace, tilingData, logicalCoreId);
                cubeKernel.Process();
            }
        }
    }
    if ASCEND_IS_AIV {
        if (tilingData.isPertoken != 0U) {
            const uint32_t logicalCoreId = AscendC::GetBlockIdx() / AscendC::GetTaskRation();
            QmmVectorKernel vectorKernel;
            vectorKernel.Init(scale, pertokenScale, y, workspace, tilingData, logicalCoreId,
                              AscendC::GetSubBlockIdx());
            vectorKernel.Process();
        }
    }
}

extern "C" __global__ __vector__ void qmm_custom_bf16_vector_kernel(
    GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y, GM_ADDR workspace,
    QmmCustomTilingData tilingData)
{
    const uint32_t vectorBlockId = AscendC::GetBlockIdx();
    const uint32_t logicalCoreId = vectorBlockId / kVectorSubcoreCount;
    const uint32_t subBlockId = vectorBlockId % kVectorSubcoreCount;
    QmmVectorKernel vectorKernel;
    vectorKernel.Init(scale, pertokenScale, y, workspace, tilingData, logicalCoreId, subBlockId);
    vectorKernel.ProcessStandaloneBf16VectorPhase();
}

#ifndef ACL_PROF_TENSOR_INFO_DEFINED
#define ACL_PROF_TENSOR_INFO_DEFINED
struct aclprofTensor {
    uint32_t type;
    uint32_t format;
    uint32_t dataType;
    uint32_t shapeDim;
    uint32_t shape[8];
};

struct aclprofTensorInfo {
    uint64_t opNameId;
    uint64_t opTypeId;
    uint32_t resv;
    uint32_t tensorNum;
    uint32_t kernelType;
    uint32_t blockNums;
    void *stream;
    aclprofTensor *tensors;
};

struct aclprofEventAttributes {
    uint16_t version;
    uint16_t size;
    uint32_t messageType;
    union message {
        aclprofTensorInfo *tensorInfo;
    } message;
};
#endif

namespace ascendc_ops {
aclDataType GetAclDataType(at::ScalarType scalarType)
{
    switch (scalarType) {
        case at::ScalarType::Float:
            return ACL_FLOAT;
        case at::ScalarType::Int:
            return ACL_INT32;
        case at::ScalarType::BFloat16:
            return ACL_BF16;
        case at::ScalarType::Char:
            return ACL_INT8;
        default:
            TORCH_CHECK(false, "qmm_custom: unsupported profiling dtype ", scalarType);
    }
}

aclprofTensor MakeProfileTensor(const at::Tensor &tensor, uint32_t type, aclFormat format)
{
    TORCH_CHECK(tensor.dim() <= 8, "qmm_custom: profiling supports at most 8 dimensions");
    aclprofTensor result = {};
    result.type = type;
    result.format = static_cast<uint32_t>(format);
    result.dataType = static_cast<uint32_t>(GetAclDataType(tensor.scalar_type()));
    result.shapeDim = static_cast<uint32_t>(tensor.dim());
    for (int64_t index = 0; index < tensor.dim(); ++index) {
        TORCH_CHECK(tensor.size(index) >= 0 &&
                        static_cast<uint64_t>(tensor.size(index)) <= UINT32_MAX,
                    "qmm_custom: profiling dimension is out of uint32 range");
        result.shape[index] = static_cast<uint32_t>(tensor.size(index));
    }
    return result;
}

void CheckNpuTensor(const at::Tensor &tensor, const char *name)
{
    TORCH_CHECK(tensor.defined(), "qmm_custom: ", name, " must be defined");
    TORCH_CHECK(tensor.device().type() == c10::DeviceType::PrivateUse1,
                "qmm_custom: ", name, " must be an NPU tensor");
}

at::Tensor qmm_custom(const at::Tensor &x1, const at::Tensor &x2, const at::Tensor &scale,
                      const c10::optional<at::Tensor> &pertokenScale)
{
    CheckNpuTensor(x1, "x1");
    CheckNpuTensor(x2, "x2");
    CheckNpuTensor(scale, "scale");
    const bool isPertoken = pertokenScale.has_value() && pertokenScale->defined();
    if (isPertoken) {
        CheckNpuTensor(*pertokenScale, "pertoken_scale");
    }
    TORCH_CHECK(x1.scalar_type() == at::ScalarType::Char &&
                    x2.scalar_type() == at::ScalarType::Char,
                "qmm_custom: x1 and x2 must have dtype torch.int8");
    TORCH_CHECK(scale.scalar_type() == at::ScalarType::Float,
                "qmm_custom: scale must have dtype torch.float32");
    TORCH_CHECK(!isPertoken || pertokenScale->scalar_type() == at::ScalarType::Float,
                "qmm_custom: pertoken_scale must have dtype torch.float32");
    TORCH_CHECK(x1.dim() == 2 && x2.dim() == 2,
                "qmm_custom: x1 and x2 must be 2-D tensors");
    TORCH_CHECK(scale.dim() == 1, "qmm_custom: scale must have shape [N]");
    TORCH_CHECK(!isPertoken || pertokenScale->dim() == 1,
                "qmm_custom: pertoken_scale must have shape [M]");
    TORCH_CHECK(x1.device() == x2.device() && x1.device() == scale.device() &&
                    (!isPertoken || x1.device() == pertokenScale->device()),
                "qmm_custom: all inputs must be on the same NPU device");

    const uint64_t m64 = static_cast<uint64_t>(x1.size(0));
    const uint64_t k64 = static_cast<uint64_t>(x1.size(1));
    const uint64_t n64 = static_cast<uint64_t>(x2.size(1));
    TORCH_CHECK(IsSupportedShape(m64, k64, n64),
                "qmm_custom: unsupported shape M=", m64, ", K=", k64, ", N=", n64);
    TORCH_CHECK(static_cast<uint64_t>(x2.size(0)) == k64,
                "qmm_custom: x2 shape must be [K, N]");
    TORCH_CHECK(static_cast<uint64_t>(scale.numel()) == n64,
                "qmm_custom: scale shape must be [N]");
    TORCH_CHECK(!isPertoken || static_cast<uint64_t>(pertokenScale->numel()) == m64,
                "qmm_custom: pertoken_scale shape must be [M]");
    TORCH_CHECK(x1.is_contiguous() && scale.is_contiguous() &&
                    (!isPertoken || pertokenScale->is_contiguous()),
                "qmm_custom: x1 and scale tensors must be contiguous");

    const uint32_t m = static_cast<uint32_t>(m64);
    const uint32_t n = static_cast<uint32_t>(n64);
    const uint32_t k = static_cast<uint32_t>(k64);
    QmmCustomTilingData tilingData = CalcQmmTiling(isPertoken ? 1U : 0U, m, n, k);
    if (std::getenv("QMM_CUSTOM_DEBUG_TILING") != nullptr) {
        std::cerr << "QMM_TILING"
                  << " m=" << m << " k=" << k << " n=" << n
                  << " isPertoken=" << tilingData.isPertoken
                  << " usedCoreNum=" << tilingData.usedCoreNum
                  << " mCoreBlocks=" << tilingData.mCoreBlocks
                  << " nCoreBlocks=" << tilingData.nCoreBlocks
                  << " singleCoreM=" << tilingData.cubeTilingData.singleCoreM
                  << " singleCoreN=" << tilingData.cubeTilingData.singleCoreN
                  << " vectorTileCols=" << tilingData.vectorTileCols
                  << " systemWorkspaceBytes=" << tilingData.systemWorkspaceBytes
                  << " intermediateOffsetBytes=" << tilingData.intermediateOffsetBytes
                  << " intermediateBytes=" << tilingData.intermediateBytes
                  << " totalWorkspaceBytes=" << tilingData.totalWorkspaceBytes
                  << std::endl;
    }
    TORCH_CHECK(tilingData.workspaceBaseIsFullAllocation == 1U,
                "qmm_custom: unexpected workspace ABI");

    const at::ScalarType outputType =
        isPertoken ? at::ScalarType::BFloat16 : at::ScalarType::Int;
    const uint64_t outputElements = static_cast<uint64_t>(m) * n;
    const uint64_t outputElementBytes = isPertoken ? sizeof(uint16_t) : sizeof(int32_t);
    const uint64_t outputBytes = outputElements * outputElementBytes;
    const uint64_t workspaceOffset = AlignUp64(outputBytes, kWorkspaceAlignment);
    TORCH_CHECK(workspaceOffset <= UINT64_MAX - tilingData.totalWorkspaceBytes,
                "qmm_custom: output/workspace allocation overflow");
    const uint64_t allocationBytes = workspaceOffset + tilingData.totalWorkspaceBytes;
    const uint64_t allocationElements =
        (allocationBytes + outputElementBytes - 1U) / outputElementBytes;
    TORCH_CHECK(allocationElements <= static_cast<uint64_t>(INT64_MAX),
                "qmm_custom: backing allocation is too large");

    // y is a view of this allocation, so its storage owns workspace until all asynchronous work using y ends.
    at::Tensor backing = at::empty({static_cast<int64_t>(allocationElements)},
                                   x1.options().dtype(outputType));
    at::Tensor y = backing.narrow(0, 0, static_cast<int64_t>(outputElements))
                       .view({static_cast<int64_t>(m), static_cast<int64_t>(n)});
    auto *backingBase = static_cast<uint8_t *>(backing.mutable_data_ptr());
    uint8_t *workspace = backingBase + workspaceOffset;
    uint8_t *pertokenPtr =
        isPertoken ? static_cast<uint8_t *>(pertokenScale->mutable_data_ptr()) : nullptr;

    const auto npuStream = c10_npu::getCurrentNPUStream();
    auto aclStream = npuStream.stream(true);
    std::vector<aclprofTensor> tensors;
    tensors.reserve(isPertoken ? 5U : 4U);
    tensors.push_back(MakeProfileTensor(x1, 0U, ACL_FORMAT_ND));
    tensors.push_back(MakeProfileTensor(x2, 0U, ACL_FORMAT_FRACTAL_NZ));
    tensors.push_back(MakeProfileTensor(scale, 0U, ACL_FORMAT_ND));
    if (isPertoken) {
        tensors.push_back(MakeProfileTensor(*pertokenScale, 0U, ACL_FORMAT_ND));
    }
    tensors.push_back(MakeProfileTensor(y, 1U, ACL_FORMAT_ND));
    aclprofTensorInfo tensorInfo = {aclprofStr2Id("qmm_custom"), aclprofStr2Id("QmmCustom"),
                                    0U, static_cast<uint32_t>(tensors.size()), 0U,
                                    tilingData.usedCoreNum, aclStream, tensors.data()};
    aclprofEventAttributes attributes = {
        1U, static_cast<uint16_t>(sizeof(aclprofEventAttributes::message)), 0U, {&tensorInfo}};
    aclprofRangePushEx(&attributes);
    if (isPertoken) {
        // Direct launches use stream ordering as the Cube-to-Vector barrier for every BF16 shape.
        // This avoids sharing CrossCore flag IDs with the Matmul high-level API, which the CANN
        // documentation explicitly warns can conflict on Atlas A2.
        QmmCustomTilingData cubePhaseTiling = tilingData;
        cubePhaseTiling.isPertoken = 0U;
        uint8_t *intermediate = workspace + tilingData.intermediateOffsetBytes;
        qmm_custom_kernel<<<tilingData.usedCoreNum, 0, aclStream>>>(
            static_cast<uint8_t *>(x1.mutable_data_ptr()),
            static_cast<uint8_t *>(x2.mutable_data_ptr()),
            static_cast<uint8_t *>(scale.mutable_data_ptr()), pertokenPtr,
            intermediate, workspace, cubePhaseTiling);
        const uint32_t vectorBlockNum = tilingData.usedCoreNum * kVectorSubcoreCount;
        qmm_custom_bf16_vector_kernel<<<vectorBlockNum, 0, aclStream>>>(
            static_cast<uint8_t *>(scale.mutable_data_ptr()), pertokenPtr,
            static_cast<uint8_t *>(y.mutable_data_ptr()), workspace, tilingData);
    } else {
        qmm_custom_kernel<<<tilingData.usedCoreNum, 0, aclStream>>>(
            static_cast<uint8_t *>(x1.mutable_data_ptr()),
            static_cast<uint8_t *>(x2.mutable_data_ptr()),
            static_cast<uint8_t *>(scale.mutable_data_ptr()), pertokenPtr,
            static_cast<uint8_t *>(y.mutable_data_ptr()), workspace, tilingData);
    }
    aclprofRangePop();
    return y;
}
} // namespace ascendc_ops

TORCH_LIBRARY(ascendc_ops, m)
{
    m.def("qmm_custom(Tensor x1, Tensor x2, Tensor scale, Tensor? pertoken_scale=None) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m)
{
    m.impl("qmm_custom", TORCH_FN(ascendc_ops::qmm_custom));
}


Overwriting /mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/qmm_custom/qmm_custom.asc


---
## 4 编译自定义算子

### 任务四：成功编译自定义算子


在算子目录下执行编译命令：

In [4]:
!cd {QMM_CUSTOM_DIR} && rm -rf build && cmake -S . -B build -DCMAKE_ASC_ARCHITECTURES=dav-2201 && cmake --build build -j


-- System processer: aarch64
-- CMAKE_ASC_COMPILER: /home/developer/Ascend/cann-9.0.0/bin/bisheng
-- ASCEND_CANN_PACKAGE_LINUX_PATH: /home/developer/Ascend/cann-9.0.0/aarch64-linux
-- CMAKE_ASC_LLD_LINKER: /home/developer/Ascend/cann-9.0.0/aarch64-linux/ccec_compiler/bin/ld.lld


-- The CXX compiler identification is GNU 9.4.0
-- Detecting CXX compiler ABI info


-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done


-- Found Python3: /opt/buildtools/Python-3.11.4/bin/python3 (found version "3.11.4") found components: Interpreter Development Development.Module Development.Embed 


CMake Warning at /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  static library kineto_LIBRARY-NOTFOUND not found.
Call Stack (most recent call first):
  /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:125 (append_torchlib_if_found)
  CMakeLists.txt:14 (find_package)


-- Found Torch: /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch/lib/libtorch.so  


...

1 warning generated.


/mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/qmm_custom/qmm_custom.asc:506:18: warning: 'SetSysWorkspace' is deprecated: NOTICE: SetSysWorkSpace has been deprecated and will be removed in the next version. [-Wdeprecated-declarations]
        AscendC::SetSysWorkspace(workspace);
                 ^
/home/developer/Ascend/cann-9.0.0/aarch64-linux/asc/impl/basic_api/dav_c220/kernel_operator_common_impl.h:28:3: note: 'SetSysWorkspace' has been explicitly marked deprecated here
[[deprecated(
  ^


1 warning generated.


[100%] Linking ASC shared library libascendc_ops.so


[100%] Built target ascendc_ops


编译成功后会在 `build` 目录下生成 `libascendc_ops.so`。


---
## 5 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale_npu)
```

### 5.1 测试算子功能

需要测试的 cases 的 shape 如下，对应的是 Qwen3-8B 模型中实际出现的调用，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [5]:
import subprocess
import sys

QMM_CUSTOM_SO = QMM_CUSTOM_DIR / 'build/libascendc_ops.so'
QMM_TEST_SCRIPT = QMM_CUSTOM_DIR / 'torch_qmm_24case_test.py'
assert QMM_CUSTOM_SO.is_file(), QMM_CUSTOM_SO
assert QMM_TEST_SCRIPT.is_file(), QMM_TEST_SCRIPT
subprocess.run(
    [
        sys.executable,
        str(QMM_TEST_SCRIPT),
        '--library', str(QMM_CUSTOM_SO),
        '--case', 'all',
        '--mode', 'all',
    ],
    check=True,
    cwd=str(QMM_CUSTOM_DIR),
)


/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")


STAGE PREPARE case=01 mode=int32 M=1 K=4096 N=4096
STAGE FORMAT_CAST case=01 mode=int32 M=1 K=4096 N=4096


FORMAT case=01 mode=int32 M=1 K=4096 N=4096 expected=29 actual=29
STAGE REFERENCE case=01 mode=int32 M=1 K=4096 N=4096
STAGE CUSTOM case=01 mode=int32 M=1 K=4096 N=4096


STAGE SYNCHRONIZE case=01 mode=int32 M=1 K=4096 N=4096
STAGE D2H case=01 mode=int32 M=1 K=4096 N=4096
STAGE COMPARE case=01 mode=int32 M=1 K=4096 N=4096
METRICS case=01 mode=int32 M=1 K=4096 N=4096 exact=true bad_count=0
RESULT case=01 mode=int32 M=1 K=4096 N=4096 status=PASS exit_code=0
STAGE PREPARE case=02 mode=bf16 M=1 K=4096 N=4096
STAGE FORMAT_CAST case=02 mode=bf16 M=1 K=4096 N=4096
FORMAT case=02 mode=bf16 M=1 K=4096 N=4096 expected=29 actual=29
STAGE REFERENCE case=02 mode=bf16 M=1 K=4096 N=4096
STAGE CUSTOM case=02 mode=bf16 M=1 K=4096 N=4096
STAGE SYNCHRONIZE case=02 mode=bf16 M=1 K=4096 N=4096
STAGE D2H case=02 mode=bf16 M=1 K=4096 N=4096
STAGE COMPARE case=02 mode=bf16 M=1 K=4096 N=4096
METRICS case=02 mode=bf16 M=1 K=4096 N=4096 rtol=0.01 atol=0.01 max_abs=0 max_rel=0 bad_count=0 bad_rows=0 first_bad_row=-1 last_bad_row=-1


RESULT case=02 mode=bf16 M=1 K=4096 N=4096 status=PASS exit_code=0
STAGE PREPARE case=03 mode=int32 M=1 K=4096 N=6144
STAGE FORMAT_CAST case=03 mode=int32 M=1 K=4096 N=6144
FORMAT case=03 mode=int32 M=1 K=4096 N=6144 expected=29 actual=29
STAGE REFERENCE case=03 mode=int32 M=1 K=4096 N=6144
STAGE CUSTOM case=03 mode=int32 M=1 K=4096 N=6144
STAGE SYNCHRONIZE case=03 mode=int32 M=1 K=4096 N=6144
STAGE D2H case=03 mode=int32 M=1 K=4096 N=6144
STAGE COMPARE case=03 mode=int32 M=1 K=4096 N=6144
METRICS case=03 mode=int32 M=1 K=4096 N=6144 exact=true bad_count=0
RESULT case=03 mode=int32 M=1 K=4096 N=6144 status=PASS exit_code=0
STAGE PREPARE case=04 mode=bf16 M=1 K=4096 N=6144
STAGE FORMAT_CAST case=04 mode=bf16 M=1 K=4096 N=6144
FORMAT case=04 mode=bf16 M=1 K=4096 N=6144 expected=29 actual=29
STAGE REFERENCE case=04 mode=bf16 M=1 K=4096 N=6144
STAGE CUSTOM case=04 mode=bf16 M=1 K=4096 N=6144
STAGE SYNCHRONIZE case=04 mode=bf16 M=1 K=4096 N=6144
STAGE D2H case=04 mode=bf16 M=1 K=4096 N=6144

...

METRICS case=22 mode=bf16 M=4096 K=4096 N=24576 rtol=0.01 atol=0.01 max_abs=0 max_rel=0 bad_count=0 bad_rows=0 first_bad_row=-1 last_bad_row=-1
RESULT case=22 mode=bf16 M=4096 K=4096 N=24576 status=PASS exit_code=0
STAGE PREPARE case=23 mode=int32 M=4096 K=12288 N=4096
STAGE FORMAT_CAST case=23 mode=int32 M=4096 K=12288 N=4096
FORMAT case=23 mode=int32 M=4096 K=12288 N=4096 expected=29 actual=29
STAGE REFERENCE case=23 mode=int32 M=4096 K=12288 N=4096
STAGE CUSTOM case=23 mode=int32 M=4096 K=12288 N=4096
STAGE SYNCHRONIZE case=23 mode=int32 M=4096 K=12288 N=4096
STAGE D2H case=23 mode=int32 M=4096 K=12288 N=4096


STAGE COMPARE case=23 mode=int32 M=4096 K=12288 N=4096
METRICS case=23 mode=int32 M=4096 K=12288 N=4096 exact=true bad_count=0


RESULT case=23 mode=int32 M=4096 K=12288 N=4096 status=PASS exit_code=0
STAGE PREPARE case=24 mode=bf16 M=4096 K=12288 N=4096
STAGE FORMAT_CAST case=24 mode=bf16 M=4096 K=12288 N=4096
FORMAT case=24 mode=bf16 M=4096 K=12288 N=4096 expected=29 actual=29
STAGE REFERENCE case=24 mode=bf16 M=4096 K=12288 N=4096
STAGE CUSTOM case=24 mode=bf16 M=4096 K=12288 N=4096
STAGE SYNCHRONIZE case=24 mode=bf16 M=4096 K=12288 N=4096
STAGE D2H case=24 mode=bf16 M=4096 K=12288 N=4096
STAGE COMPARE case=24 mode=bf16 M=4096 K=12288 N=4096
METRICS case=24 mode=bf16 M=4096 K=12288 N=4096 rtol=0.01 atol=0.01 max_abs=0 max_rel=0 bad_count=0 bad_rows=0 first_bad_row=-1 last_bad_row=-1


RESULT case=24 mode=bf16 M=4096 K=12288 N=4096 status=PASS exit_code=0
SUMMARY TOTAL=24 PASS=24 FAIL=0 EXECUTED=24


CompletedProcess(args=['/mnt/workspace/qwen3_8b_tutorial_run/v1/venv-recipes/bin/python', '/mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/qmm_custom/torch_qmm_24case_test.py', '--library', '/mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/qmm_custom/build/libascendc_ops.so', '--case', 'all', '--mode', 'all'], returncode=0)

### 5.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [6]:
import pandas as pd
import subprocess
import sys

QMM_PROFILE_SCRIPT = QMM_CUSTOM_DIR / 'torch_qmm_profile.py'
QMM_PROFILE_OUTPUT = SRC_DIR / 'prof_output'
subprocess.run(
    [
        sys.executable,
        str(QMM_PROFILE_SCRIPT),
        '--library', str(QMM_CUSTOM_SO),
        '--output', str(QMM_PROFILE_OUTPUT),
        '--warmup', '1',
    ],
    check=True,
    cwd=str(QMM_CUSTOM_DIR),
)
profile_summary = pd.read_csv(QMM_PROFILE_OUTPUT / 'qmm_operator_duration_summary.csv')
display(profile_summary)


/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")


PROFILED mode=int32 M=1 K=4096 N=4096 dtype=torch.int32 shape=(1, 4096)
PROFILED mode=bf16 M=1 K=4096 N=4096 dtype=torch.bfloat16 shape=(1, 4096)


PROFILED mode=int32 M=1 K=4096 N=6144 dtype=torch.int32 shape=(1, 6144)
PROFILED mode=bf16 M=1 K=4096 N=6144 dtype=torch.bfloat16 shape=(1, 6144)


PROFILED mode=int32 M=1 K=4096 N=24576 dtype=torch.int32 shape=(1, 24576)


PROFILED mode=bf16 M=1 K=4096 N=24576 dtype=torch.bfloat16 shape=(1, 24576)


...

[2026-07-23 22:31:17] [INFO] [378634] profiler.py: Start parsing profiling data in sync mode at: /mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/prof_output/bd5bbcfa4e2f45c2a56d0c5b4f2a55a3_377768_20260723223110149_ascend_pt
|

/

-[2026-07-23 22:31:19] [INFO] [378660] profiler.py: CANN profiling data parsed in a total time of 0:00:02.018180


[2026-07-23 22:31:20] [INFO] [378634] profiler.py: All profiling data parsed in a total time of 0:00:02.991728
PROFILE_SUMMARY {"library": "/mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/qmm_custom/build/libascendc_ops.so", "output": "/mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/prof_output", "shape_count": 12, "expected_invocations": 24, "kernel_details_csv": ["/mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/prof_output/bd5bbcfa4e2f45c2a56d0c5b4f2a55a3_377768_20260723223110149_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv"], "qmm_row_count": 36, "duration_summary_rows": 12}


,M,K,N,INT32_Cube_us,BF16_Cube_us,BF16_Vector_us,BF16_Total_us
0,1,4096,4096,21.96,19.24,6.04,25.28
1,1,4096,6144,27.10,25.92,7.88,33.80
2,1,4096,24576,93.94,86.46,7.90,94.36
3,1,12288,4096,35.52,34.82,6.16,40.98
4,50,4096,4096,21.64,19.92,11.70,31.62
5,50,4096,6144,25.82,27.96,11.42,39.38
6,50,4096,24576,97.26,93.56,20.72,114.28
7,50,12288,4096,63.30,61.96,11.24,73.20
8,4096,4096,4096,383.26,385.68,187.88,573.56
9,4096,4096,6144,597.72,596.94,221.52,818.46


---
## 6 自定义算子接入模型

### 6.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.ascendc_ops.qmm_custom` 算子。

查看原始实现：

In [7]:
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
for number, line in enumerate(W8A8_FILE.read_text(encoding='utf-8').splitlines(), 1):
    if 'npu_quant_matmul(' in line or 'torch.ops.ascendc_ops.qmm_custom(' in line:
        print(f'{number}: {line.strip()}')


133: x = torch.ops.ascendc_ops.qmm_custom(


### 6.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.ascendc_ops.qmm_custom`。

替换后的方法为：

```python
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight, 
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [8]:
# 6.2 写入已通过全模型验证的 QmmCustom 集成实现
import os
import torch
import torch_npu

W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
QMM_CUSTOM_SO = QMM_CUSTOM_DIR / 'build/libascendc_ops.so'
CUSTOM_W8A8_SOURCE = '# coding=utf-8\n# Adapted from\n# https://github.com/vllm-project/vllm/blob/v0.9.0/vllm/model_executor/layers/quantization/compressed_tensors/schemes/compressed_tensors_w8a8_int8.py\n# Copyright (c) 2025-2026 Huawei Technologies Co., Ltd.\n# SPDX-FileCopyrightText: Copyright contributors to the vLLM project\n#\n# Licensed under the Apache License, Version 2.0 (the "License");\n# you may not use this file except in compliance with the License.\n# You may obtain a copy of the License at\n#\n#     http://www.apache.org/licenses/LICENSE-2.0\n#\n# Unless required by applicable law or agreed to in writing, software\n# distributed under the License is distributed on an "AS IS" BASIS,\n# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n# See the License for the specific language governing permissions and\n# limitations under the License.\n\nimport atexit\nimport os\nfrom typing import Callable, Any, Dict, List, Optional, Union\n\nimport torch\nfrom torch.nn import Parameter\nimport torch_npu\n\nfrom compressed_tensors.quantization import QuantizationStrategy\nfrom module.quantization.compressed_tensors.compressed_tensors_scheme import CompressedTensorsScheme\nfrom module.utils import set_weight_attrs\n\n\nBEFORE_INIT = 0\nAFTER_INIT = 1\n_QMM_CUSTOM_LOADED = False\n_QMM_CUSTOM_CALLS = {"int32": 0, "bf16": 0}\n\n\ndef _ensure_qmm_custom_loaded() -> None:\n    global _QMM_CUSTOM_LOADED\n    if _QMM_CUSTOM_LOADED:\n        return\n    library_path = os.environ.get("QMM_CUSTOM_SO_PATH")\n    if not library_path:\n        raise RuntimeError("QMM_CUSTOM_SO_PATH must point to libascendc_ops.so")\n    if not os.path.isfile(library_path):\n        raise FileNotFoundError(f"QmmCustom library not found: {library_path}")\n    torch.ops.load_library(library_path)\n    if not hasattr(torch.ops, "ascendc_ops") or not hasattr(torch.ops.ascendc_ops, "qmm_custom"):\n        raise RuntimeError(f"QmmCustom was not registered after loading {library_path}")\n    _QMM_CUSTOM_LOADED = True\n    print(f"QMM_CUSTOM_LIBRARY_LOADED={library_path}", flush=True)\n\n\ndef _report_qmm_custom_calls() -> None:\n    total = _QMM_CUSTOM_CALLS["int32"] + _QMM_CUSTOM_CALLS["bf16"]\n    if total:\n        print(\n            "QMM_CUSTOM_CALL_COUNT "\n            f"total={total} int32={_QMM_CUSTOM_CALLS[\'int32\']} bf16={_QMM_CUSTOM_CALLS[\'bf16\']}",\n            flush=True,\n        )\n\n\natexit.register(_report_qmm_custom_calls)\n\n\nclass CompressedTensorsW8A8Int8LinearMethod(CompressedTensorsScheme):\n    _kernel_backends_being_used: set[str] = set()\n\n    def __init__(self, strategy: str, is_static_input_scheme: bool,\n                 input_symmetric: bool):\n        self.strategy = strategy\n        self.is_static_input_scheme = is_static_input_scheme\n        self.input_symmetric = input_symmetric\n\n\n    def create_weights(self, layer: torch.nn.Module,\n                       output_partition_sizes: List[int],\n                       input_size_per_partition: int,\n                       params_dtype: torch.dtype,\n                       weight_loader: Callable,\n                       **kwargs):\n        self.logical_widths = output_partition_sizes\n\n        weight = Parameter(torch.empty(sum(output_partition_sizes),\n                                       input_size_per_partition,\n                                       dtype=torch.int8),\n                           requires_grad=False)\n        set_weight_attrs(weight, {"input_dim": 1, "output_dim": 0, "weight_loader": weight_loader})\n\n        layer.register_parameter("weight", weight)\n        scale_dtype = torch.float32 if params_dtype == torch.float16 else torch.bfloat16\n\n\n        if self.strategy == QuantizationStrategy.TENSOR:\n            weight_offset = None\n            weight_scale = Parameter(torch.empty(len(output_partition_sizes), dtype=scale_dtype), requires_grad=False)\n            set_weight_attrs(weight_scale, {"weight_loader": weight_loader})\n        else:\n            weight_scale = Parameter(torch.empty((sum(output_partition_sizes), 1), dtype=scale_dtype),\n                                     requires_grad=False)\n            set_weight_attrs(weight_scale, {"output_dim": 0, "weight_loader": weight_loader})\n\n        layer.register_parameter("weight_scale", weight_scale)\n        smooth_scales = Parameter(torch.ones(input_size_per_partition, dtype=scale_dtype), requires_grad=False)\n        layer.register_parameter("smooth_scales", smooth_scales)\n\n        setattr(layer, "init_state", BEFORE_INIT)\n\n        self.empty_out = torch.empty(1, dtype=params_dtype)\n\n\n    def apply_weights(self, layer: torch.nn.Module,\n                    x: torch.Tensor,\n                    bias: Optional[torch.Tensor],\n                    dynamic_scale: Optional = None,\n                    out_dtype: Optional = torch.bfloat16\n                    ) -> Union[torch.Tensor, Dict[str, Any]]:\n        if dynamic_scale is not None:\n            x_scale = dynamic_scale\n        else:\n            x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)\n        out_shape_dim_0 = x.size()[:-1]\n        x = x.view(-1, x.size(-1))\n        x_scale = x_scale.view(-1)\n        _ensure_qmm_custom_loaded()\n        if bias is not None or layer.bias is not None:\n            raise NotImplementedError("QmmCustom integration does not support quantized Linear bias")\n        weight_scale = layer.weight_scale.view(-1)\n        if weight_scale.dtype != torch.float32:\n            raise TypeError(f"QmmCustom requires FP32 weight_scale, got {weight_scale.dtype}")\n        pertoken_scale = None if out_dtype == torch.int32 else x_scale\n        x = torch.ops.ascendc_ops.qmm_custom(\n            x,\n            layer.weight,\n            weight_scale,\n            pertoken_scale=pertoken_scale,\n        )\n        _QMM_CUSTOM_CALLS["int32" if out_dtype == torch.int32 else "bf16"] += 1\n        x = x.view(out_shape_dim_0 + (-1, ))\n        if out_dtype == torch.int32:\n            return x, x_scale\n        else:\n            return x\n\n    def process_weights_after_loading(self, layer, is_transpose=True, is_nz=True, scales_dtype=None):\n        weight = layer.weight\n        weight_scale = layer.weight_scale\n        smooth_scales = layer.smooth_scales\n        if is_transpose:\n            weight.data = weight.data.transpose(-2, -1)\n        if is_nz:\n            weight.data = torch_npu.npu_format_cast(weight.data.contiguous(), 29)  # 29: format nz\n        if \'scale_dtype\' in scales_dtype:\n            weight_scale.data = weight_scale.data.to(scales_dtype.get(\'scale_dtype\'))\n        if \'smooth_scale_dtype\' in scales_dtype:\n            smooth_scales.data = smooth_scales.data.to(scales_dtype.get(\'smooth_scale_dtype\'))\n        layer.weight = Parameter(weight, requires_grad=False)\n        layer.weight_scale = Parameter(weight_scale, requires_grad=False)\n        layer.smooth_scales = Parameter(smooth_scales, requires_grad=False)\n'
W8A8_FILE.write_text(CUSTOM_W8A8_SOURCE, encoding='utf-8')
os.environ['QMM_CUSTOM_SO_PATH'] = str(QMM_CUSTOM_SO)
torch.ops.load_library(str(QMM_CUSTOM_SO))

result = W8A8_FILE.read_text(encoding='utf-8')
assert 'torch.ops.ascendc_ops.qmm_custom' in result
assert 'torch_npu.npu_quant_matmul(' not in result
assert hasattr(torch.ops.ascendc_ops, 'qmm_custom')
print('[接入] 已写入:', W8A8_FILE)
print('[动态库] 已加载:', QMM_CUSTOM_SO)
print('[验证] torch.ops.ascendc_ops.qmm_custom 已注册')


/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")


[接入] 已写入: /mnt/workspace/qwen3_8b_tutorial_final/v1/src/inference_scripts/recipe_qwen3_8b/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py
[动态库] 已加载: /mnt/workspace/qwen3_8b_tutorial_final/v1/src/op_custom/qmm_custom/build/libascendc_ops.so
[验证] torch.ops.ascendc_ops.qmm_custom 已注册


/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/__init__.py:324: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


### 6.3 替换实现后执行量化推理

指定本地的量化模型权重路径，调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，对比替换前的生成文本是否一致。

**注意：需要先完成第五章的量化模型推理过程，并将`W8A8_WEIGHT`指定为本地的量化模型权重的路径**

In [9]:
import os
import subprocess
import sys
from pathlib import Path
from inference_scripts.recipe_workflow import prepare_runtime_yaml

W8A8_WEIGHT = Path(os.environ['QWEN3_W8A8_WEIGHT'])
assert W8A8_WEIGHT.is_dir(), W8A8_WEIGHT
YAML_A8W8_TEMPLATE = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8_CUSTOM = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom.yaml'
prepare_runtime_yaml(YAML_A8W8_TEMPLATE, YAML_A8W8_CUSTOM, model_path=str(W8A8_WEIGHT))

env = os.environ.copy()
env['PATH'] = os.path.dirname(sys.executable) + ':' + env.get('PATH', '')
env['QMM_CUSTOM_SO_PATH'] = str(QMM_CUSTOM_SO)
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', str(YAML_A8W8_CUSTOM)],
    cwd=str(RECIPE_ROOT), env=env, check=True,
)
print('[模型推理] 返回码 =', result.returncode)


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/qwen3_8b_tutorial_final/v1/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom.yaml)
world_size = 1


/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")


[INFO][2026-07-23 22:31:37.525][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-23 22:31:37.525][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-23 22:31:37.525][patch_getenv.py:15] get env LANG = C.UTF-8
[INFO][2026-07-23 22:31:37.527][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/qwen3_8b_tutorial_final/v1/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-23 22:31:37.527][patch_getenv.py:15] get env RES_PATH = res/20260723/qwen3_8b_qwen3_8b_a8w8_custom
[INFO][2026-07-23 22:31:37.527][infer.py:146] Inference Configuration
[INFO][2026-07-23 22:31:37.527][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/qwen3_8b_models/Qwen3-8B-W8A8', output_path='/mnt/workspace/qwen3_8b_tutorial_final/v1/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom', dtype='bfloat16', with_ckpt=True, next_n=0, platform_version=<PlatformVersion.A3: 'A3'>, exe_mode='eager', e

[INFO][2026-07-23 22:31:38.448][execution_engine.py:126] Loading main model...
/mnt/workspace/qwen3_8b_tutorial_run/v1/venv-recipes/lib/python3.11/site-packages/transformers/modeling_rope_utils.py:927: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(
[INFO][2026-07-23 22:31:38.532][model_worker.py:185] Try to load pretrained model in path: /mnt/workspace/qwen3_8b_models/Qwen3-8B-W8A8
[INFO][2026-07-23 22:31:38.532][comm_manager.py:442] CommManager: group 'attn_tp_group' registered (my_subgroup=[0], reuse=True, rank_in_group=0)
[INFO][2026-07-23 22:31:38.532][comm_manager.py:442] CommManager: group 'lmhead_tp_group' registered (my_subgroup=[0], reuse=True, rank_in_

`torch_dtype` is deprecated! Use `dtype` instead!
Loading safetensors checkpoint shards:   0% Completed | 0/37 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  24% Completed | 9/37 [00:00<00:00, 82.89it/s]


...

[INFO][2026-07-23 22:32:13.776][scheduler.py:518] [Main] Inference time (decode): 41.52 ms
[INFO][2026-07-23 22:32:13.818][scheduler.py:518] [Main] Inference time (decode): 41.44 ms
[INFO][2026-07-23 22:32:13.860][scheduler.py:518] [Main] Inference time (decode): 40.31 ms
[INFO][2026-07-23 22:32:13.902][scheduler.py:518] [Main] Inference time (decode): 39.96 ms
[INFO][2026-07-23 22:32:13.943][scheduler.py:518] [Main] Inference time (decode): 39.97 ms


[INFO][2026-07-23 22:32:13.985][scheduler.py:518] [Main] Inference time (decode): 40.17 ms
[INFO][2026-07-23 22:32:14.028][scheduler.py:518] [Main] Inference time (decode): 42.16 ms
[INFO][2026-07-23 22:32:14.070][scheduler.py:518] [Main] Inference time (decode): 40.10 ms
[INFO][2026-07-23 22:32:14.121][scheduler.py:518] [Main] Inference time (decode): 50.13 ms
[INFO][2026-07-23 22:32:14.163][scheduler.py:518] [Main] Inference time (decode): 40.25 ms


[INFO][2026-07-23 22:32:14.205][scheduler.py:518] [Main] Inference time (decode): 40.66 ms
[INFO][2026-07-23 22:32:14.247][scheduler.py:518] [Main] Inference time (decode): 40.44 ms
[INFO][2026-07-23 22:32:14.291][scheduler.py:518] [Main] Inference time (decode): 42.92 ms
[INFO][2026-07-23 22:32:14.333][scheduler.py:518] [Main] Inference time (decode): 40.55 ms
[INFO][2026-07-23 22:32:14.375][scheduler.py:518] [Main] Inference time (decode): 40.88 ms


[INFO][2026-07-23 22:32:14.417][scheduler.py:518] [Main] Inference time (decode): 40.61 ms
[INFO][2026-07-23 22:32:14.459][scheduler.py:518] [Main] Inference time (decode): 40.41 ms
[INFO][2026-07-23 22:32:14.501][scheduler.py:518] [Main] Inference time (decode): 40.62 ms
[INFO][2026-07-23 22:32:14.543][scheduler.py:518] [Main] Inference time (decode): 40.49 ms
[INFO][2026-07-23 22:32:14.543][infer.py:99] Request 0: outputs: The output of an **attention function** is typically a **weighted sum** of the **value vectors**, where the weights are determined by the **similarity** between the **query vector** and the **key vectors** from the set of key-value pairs.

### Formal Description:

Given:
- A **query vector** $ Q $
- A set of **key-value pairs**: $ (K_1, V_1), (K_2, V_2), \dots, (K_n, V_n) $

The **attention function** computes the output as:

$$
\text{Attention}(Q, K, V) = \sum_{i=1}^{n} \alpha_i V_i
$$

Where:
- $ \alpha_i = \text{softmax}\left( \frac{Q \cdot K_i}{\sqrt{d_k}} \rig

[模型推理] 返回码 = 0


### 实测文本对比说明

在 Huawei Developer Space 上，原生 `npu_quant_matmul` 与 QmmCustom 均完成 256 个 Decode step。两份输出都正确解释了 Attention 的 weighted sum、query/key/value 与 softmax；由于量化舍入导致自回归早期 Token 分叉，文本**语义一致但非逐字符一致**。完整文本与机器可读比较见：

- `integration/results/stage03_baseline/baseline-summary.json`
- `integration/results/stage03_custom/custom-summary.json`
- `integration/results/stage03_custom/baseline-vs-custom.json`


---
## 7 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 7.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [10]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_A8W8_CUSTOM_PROFILER = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml'
prepare_profiling_yaml(YAML_A8W8_CUSTOM, YAML_A8W8_CUSTOM_PROFILER, enable_profiler=True)
print('[Profiler YAML] 已生成:', YAML_A8W8_CUSTOM_PROFILER)


[Profiler YAML] 已生成: /mnt/workspace/qwen3_8b_tutorial_final/v1/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml


### 7.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [11]:
env['QMM_CUSTOM_SO_PATH'] = str(QMM_CUSTOM_SO)
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', str(YAML_A8W8_CUSTOM_PROFILER)],
    cwd=str(RECIPE_ROOT), env=env, check=True,
)
print('[Profiling 推理] 返回码 =', result.returncode)


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/qwen3_8b_tutorial_final/v1/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml)


world_size = 1


/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")


[INFO][2026-07-23 22:32:28.723][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-23 22:32:28.723][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-23 22:32:28.723][patch_getenv.py:15] get env LANG = C.UTF-8
[INFO][2026-07-23 22:32:28.725][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/qwen3_8b_tutorial_final/v1/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-23 22:32:28.725][patch_getenv.py:15] get env RES_PATH = res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler
[INFO][2026-07-23 22:32:28.725][infer.py:146] Inference Configuration
[INFO][2026-07-23 22:32:28.725][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/qwen3_8b_models/Qwen3-8B-W8A8', output_path='/mnt/workspace/qwen3_8b_tutorial_final/v1/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler', dtype='bfloat16', with_ckpt=True, next_n=0, platform_version=<PlatformVersion.A3: 'A3'>, e

[INFO][2026-07-23 22:32:29.652][execution_engine.py:126] Loading main model...
/mnt/workspace/qwen3_8b_tutorial_run/v1/venv-recipes/lib/python3.11/site-packages/transformers/modeling_rope_utils.py:927: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(
[INFO][2026-07-23 22:32:29.735][model_worker.py:185] Try to load pretrained model in path: /mnt/workspace/qwen3_8b_models/Qwen3-8B-W8A8
[INFO][2026-07-23 22:32:29.735][comm_manager.py:442] CommManager: group 'attn_tp_group' registered (my_subgroup=[0], reuse=True, rank_in_group=0)
[INFO][2026-07-23 22:32:29.735][comm_manager.py:442] CommManager: group 'lmhead_tp_group' registered (my_subgroup=[0], reuse=True, rank_in_

...

[INFO][2026-07-23 22:33:14.479][scheduler.py:518] [Main] Inference time (decode): 44.32 ms
[INFO][2026-07-23 22:33:14.525][scheduler.py:518] [Main] Inference time (decode): 44.46 ms
[INFO][2026-07-23 22:33:14.572][scheduler.py:518] [Main] Inference time (decode): 45.25 ms
[INFO][2026-07-23 22:33:14.618][scheduler.py:518] [Main] Inference time (decode): 44.53 ms
[INFO][2026-07-23 22:33:14.664][scheduler.py:518] [Main] Inference time (decode): 44.81 ms


[INFO][2026-07-23 22:33:14.711][scheduler.py:518] [Main] Inference time (decode): 44.75 ms
[INFO][2026-07-23 22:33:14.757][scheduler.py:518] [Main] Inference time (decode): 44.53 ms
[INFO][2026-07-23 22:33:14.803][scheduler.py:518] [Main] Inference time (decode): 44.60 ms
[INFO][2026-07-23 22:33:14.849][scheduler.py:518] [Main] Inference time (decode): 44.31 ms
[INFO][2026-07-23 22:33:14.895][scheduler.py:518] [Main] Inference time (decode): 44.38 ms


[INFO][2026-07-23 22:33:14.941][scheduler.py:518] [Main] Inference time (decode): 44.45 ms
[INFO][2026-07-23 22:33:14.987][scheduler.py:518] [Main] Inference time (decode): 44.65 ms
[INFO][2026-07-23 22:33:15.035][scheduler.py:518] [Main] Inference time (decode): 46.78 ms
[INFO][2026-07-23 22:33:15.081][scheduler.py:518] [Main] Inference time (decode): 44.09 ms
[INFO][2026-07-23 22:33:15.127][scheduler.py:518] [Main] Inference time (decode): 44.35 ms


[INFO][2026-07-23 22:33:15.173][scheduler.py:518] [Main] Inference time (decode): 44.24 ms
[INFO][2026-07-23 22:33:15.219][scheduler.py:518] [Main] Inference time (decode): 44.36 ms
[INFO][2026-07-23 22:33:15.219][infer.py:99] Request 0: outputs: The output of an **attention function** is typically a **weighted sum** of the **value vectors**, where the weights are determined by the **similarity** between the **query vector** and the **key vectors** from the set of key-value pairs.

### Formal Description:

Given:
- A **query vector** $ Q $
- A set of **key-value pairs**: $ (K_1, V_1), (K_2, V_2), \dots, (K_n, V_n) $

The **attention function** computes the output as:

$$
\text{Attention}(Q, K, V) = \sum_{i=1}^{n} \alpha_i V_i
$$

Where:
- $ \alpha_i = \text{softmax}\left( \frac{Q \cdot K_i}{\sqrt{d_k}} \right) $
- $ d_k $ is the dimensionality of the key vectors (used for scaling to prevent large values)
- $ \cdot $ denotes the dot product
- $ \text{softmax} $ ensures that the weights 

[Profiling 推理] 返回码 = 0


### 7.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [12]:
import pandas as pd
from pathlib import Path

RES_DIR = RECIPE_ROOT / 'models/qwen/res'
date_dirs = sorted([path for path in RES_DIR.iterdir() if path.is_dir()], key=lambda path: path.stat().st_mtime)
latest_date = date_dirs[-1]
case_dir = latest_date / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler'

def find_kernel_csv(phase):
    matches = sorted((case_dir / 'prof' / phase).rglob('kernel_details.csv'), key=lambda path: path.stat().st_mtime)
    return matches[-1] if matches else None

rows = []
for phase in ('prefill', 'decode'):
    csv_path = find_kernel_csv(phase)
    assert csv_path, f'{phase} kernel_details.csv 未找到'
    frame = pd.read_csv(csv_path)
    cube = frame[frame['Type'].str.contains('qmm_custom_kernel', na=False)]
    vector = frame[frame['Type'].eq('qmm_custom_bf16_vector_kernel')]
    rows.append({
        'Phase': phase.capitalize(),
        'Logical Qmm Calls': len(cube),
        'Cube Total(us)': cube['Duration(us)'].sum(),
        'BF16 Vector Calls': len(vector),
        'Vector Total(us)': vector['Duration(us)'].sum(),
        'Qmm Total(us)': cube['Duration(us)'].sum() + vector['Duration(us)'].sum(),
        'Kernel CSV': str(csv_path),
    })
summary = pd.DataFrame(rows)
display(summary)
print('完整的原生/自定义 Shape 对比见 integration/reports/stage04_model_profiling_comparison.md')


,Phase,Logical Qmm Calls,Cube Total(us),BF16 Vector Calls,Vector Total(us),Qmm Total(us),Kernel CSV
0,Prefill,144,8610.58,108,1455.40,10065.98,/mnt/workspace/qwen3_8b_tutorial_final/v1/src/...
1,Decode,432,21098.04,324,2102.84,23200.88,/mnt/workspace/qwen3_8b_tutorial_final/v1/src/...


完整的原生/自定义 Shape 对比见 integration/reports/stage04_model_profiling_comparison.md


### 实测性能结论

同配置 Profiler 对比结果：

| Phase | Native Qmm Total | QmmCustom Total | Custom / Native |
|---|---:|---:|---:|
| Prefill | 6,576.22 us | 10,047.50 us | 1.5279× |
| Decode | 19,338.98 us | 23,019.42 us | 1.1903× |

当前正确性优先版采用 Cube 与 Vector 两次 Stream 顺序发射，因此没有超过成熟的原生融合算子。完整 CSV 与 Shape 分组见 `integration/reports/stage04_model_profiling_comparison.md`。


## 8 提交要求

> 完整的提交说明参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 性能数据收集成功。
3. **文件命名**：`gitcode账号_result.ipynb`。
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）。
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译和单算子功能性能测试，并将该自定义算子接入了 Qwen3-8B 量化模型，验证功能并观测了其在模型中的性能。


### 附录
实践相关参考材料链接：
- [Ascend C 高阶api文档](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%AB%98%E9%98%B6API/%E9%AB%98%E9%98%B6API.md)
- [昇腾社区文档](https://www.hiascend.com/zh/document)
- [Ascend C 算子开发系列教程](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development)
- [Ascend C 算子功能调试](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development/07_Troubleshooting)
- [Matmul最佳实践样例](https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance/README.md)
- [CANN内置量化矩阵乘算子实现](https://gitcode.com/cann/ops-nn/tree/master/matmul/quant_batch_matmul_v3)